# Observed-attrition accuracy models

Refit the accuracy models on all assigned participants and run the Perplexity tipping-point analysis.


## Load packages

In [ ]:
library(readxl)
library(dplyr)
library(lme4)
library(emmeans)

NameError: name 'library' is not defined

## Prepare long accuracy data (same helpers as primary)

In [ ]:
ROOT <- normalizePath("..", winslash = "/")
PATH <- file.path(ROOT, "data", "Psy_Case_Vignette_Data_Prolific_observed_attrition.xlsx")
VIGNETTES <- c("A", "B", "C", "D", "E")
ARM_REF <- "4"
COUNTRY_COL <- "Country of residence"
ADJ_COVARIATES <- c("Age", "Sex", "ai_frequency", "ai_literacy_sum")
AILI_ITEMS <- paste0("aili_", sprintf("%02d", 1:12))
REVERSE_AILI_ITEMS <- c("aili_02", "aili_05", "aili_11")

df <- read_excel(PATH)

recode_aili_col <- function(col_name) {
  values <- as.numeric(df[[col_name]])
  if (col_name %in% REVERSE_AILI_ITEMS) {
    values <- 8 - values
  }
  values
}

aili_recoded <- as.data.frame(lapply(AILI_ITEMS, recode_aili_col), stringsAsFactors = FALSE)
names(aili_recoded) <- AILI_ITEMS
df$ai_literacy_sum <- apply(aili_recoded, 1, function(row) {
  if (any(is.na(row))) NA_real_ else sum(row)
})

make_long_outcome <- function(data, pre_cols, post_cols) {
  bind_rows(lapply(seq_along(VIGNETTES), function(i) {
    bind_rows(
      data %>%
        transmute(
          prolific_id = as.factor(prolific_id),
          vignette = factor(VIGNETTES[i], levels = VIGNETTES),
          study_arm = as.character(study_arm),
          sample_group = as.character(sample_group),
          country = as.character(.data[[COUNTRY_COL]]),
          Age = Age,
          Sex = as.character(Sex),
          ai_frequency = as.numeric(ai_frequency),
          ai_literacy_sum = as.numeric(ai_literacy_sum),
          post = 0L,
          correct = .data[[pre_cols[i]]]
        ),
      data %>%
        transmute(
          prolific_id = as.factor(prolific_id),
          vignette = factor(VIGNETTES[i], levels = VIGNETTES),
          study_arm = as.character(study_arm),
          sample_group = as.character(sample_group),
          country = as.character(.data[[COUNTRY_COL]]),
          Age = Age,
          Sex = as.character(Sex),
          ai_frequency = as.numeric(ai_frequency),
          ai_literacy_sum = as.numeric(ai_literacy_sum),
          post = 1L,
          correct = .data[[post_cols[i]]]
        )
    )
  })) %>%
    filter(
      !is.na(correct), !is.na(study_arm), !is.na(prolific_id),
      !is.na(country), !is.na(Age), !is.na(Sex)
    ) %>%
    mutate(
      correct = as.integer(correct),
      post = as.integer(post),
      study_arm = relevel(factor(study_arm), ref = ARM_REF),
      Sex = relevel(factor(Sex), ref = "Female")
    )
}

print_post_terms <- function(model, label) {
  cat("\n", label, "\n", sep = "")
  cat(
    "N participants:", n_distinct(model@frame$prolific_id),
    "; N observations:", nrow(model@frame), "\n"
  )
  if ("sample_group" %in% names(model@frame)) {
    # sample_group may not be in frame if not in formula; print from data attr if needed
  }
  print(summary(model))
  coef_mat <- coef(summary(model))
  cat("\nPre→post odds ratios (post = control arm; interactions = difference vs control):\n")
  post_rows <- grep("^post", rownames(coef_mat), value = TRUE)
  for (nm in post_rows) {
    b <- coef_mat[nm, "Estimate"]
    se <- coef_mat[nm, "Std. Error"]
    p <- coef_mat[nm, "Pr(>|z|)"]
    cat(sprintf(
      "  %s: OR = %.3f [%.3f, %.3f], p = %.4f\n",
      nm, exp(b), exp(b - 1.96 * se), exp(b + 1.96 * se), p
    ))
  }
  invisible(model)
}

print_post_study_arm_omnibus <- function(full_model, label, covariates = character()) {
  cov <- if (length(covariates)) paste("+", paste(covariates, collapse = " + ")) else ""
  reduced_formula <- as.formula(
    paste("correct ~ post + study_arm", cov, "+ (1 | prolific_id) + (1 | vignette)")
  )
  cat("\n", label, "— omnibus post × study_arm interaction (LRT, ML)\n", sep = "")
  reduced_model <- update(full_model, formula = reduced_formula)
  cmp <- anova(reduced_model, full_model)
  print(cmp)
  invisible(cmp)
}

GLMER_CTRL <- glmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 2e5))

PRE_CONDITION_COLS <- paste0("vig_", VIGNETTES, "_pre_condition_correct")
POST_CONDITION_COLS <- paste0("vig_", VIGNETTES, "_post_condition_correct")
PRE_NEXTSTEP_COLS <- paste0("vig_", VIGNETTES, "_pre_nextstep_correct")
POST_NEXTSTEP_COLS <- paste0("vig_", VIGNETTES, "_post_nextstep_correct")

long_condition <- make_long_outcome(df, PRE_CONDITION_COLS, POST_CONDITION_COLS)
long_triage <- make_long_outcome(df, PRE_NEXTSTEP_COLS, POST_NEXTSTEP_COLS)

long_condition_adj <- long_condition %>%
  filter(!is.na(ai_frequency), !is.na(ai_literacy_sum))
long_triage_adj <- long_triage %>%
  filter(!is.na(ai_frequency), !is.na(ai_literacy_sum))

cat("Wide sample_group counts:\n")
print(table(df$sample_group, useNA = "ifany"))

cat("\nlong_condition:", nrow(long_condition), "rows,",
    n_distinct(long_condition$prolific_id), "participants\n")
cat("  by sample_group (participants):\n")
print(long_condition %>% distinct(prolific_id, sample_group) %>% count(sample_group))
cat("  by post:\n")
print(long_condition %>% count(post))

cat("\nlong_triage:   ", nrow(long_triage), "rows,",
    n_distinct(long_triage$prolific_id), "participants\n")
cat("  by sample_group (participants):\n")
print(long_triage %>% distinct(prolific_id, sample_group) %>% count(sample_group))

cat("\nlong_condition_adj:", nrow(long_condition_adj), "rows,",
    n_distinct(long_condition_adj$prolific_id), "participants\n")
cat("(Adjusted models require AI covariates; allocated incomplete typically contribute 0 rows.)\n")

Wide sample_group counts:



allocated_incomplete             analysed 
                  56                  805 



long_condition: 8432 rows, 861 participants


  by sample_group (participants):


# A tibble: 2 × 2
  sample_group             n
  <chr>                <int>
1 allocated_incomplete    56
2 analysed               805


  by post:


... (2 outputs left out)

  by sample_group (participants):


# A tibble: 2 × 2
  sample_group             n
  <chr>                <int>
1 allocated_incomplete    56
2 analysed               805



long_condition_adj: 7660 rows, 766 participants


(Adjusted models require AI covariates; allocated incomplete typically contribute 0 rows.)


### Fit the unadjusted condition model

In [ ]:
condition_unadj <- glmer(
  correct ~ post * study_arm + (1 | prolific_id) + (1 | vignette),
  data = long_condition,
  family = binomial,
  control = glmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 2e5))
)

print_post_terms(
  condition_unadj,
  "Observed-attrition SA — Condition identification (unadjusted)"
)
print_post_study_arm_omnibus(
  condition_unadj,
  "Observed-attrition SA — Condition identification (unadjusted)"
)


Observed-attrition SA — Condition identification (unadjusted)
N participants: 861 ; N observations: 8432 
Generalized linear mixed model fit by maximum likelihood (Laplace
  Approximation) [glmerMod]
 Family: binomial  ( logit )
Formula: correct ~ post * study_arm + (1 | prolific_id) + (1 | vignette)
   Data: long_condition
Control: glmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 2e+05))

      AIC       BIC    logLik -2*log(L)  df.resid 
  10768.7   10839.1   -5374.3   10748.7      8422 

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-2.9245 -0.8040  0.4055  0.7563  2.7422 

Random effects:
 Groups      Name        Variance Std.Dev.
 prolific_id (Intercept) 0.8865   0.9416  
 vignette    (Intercept) 0.1972   0.4441  
Number of obs: 8432, groups:  prolific_id, 861; vignette, 5

Fixed effects:
                Estimate Std. Error z value Pr(>|z|)    
(Intercept)     -0.06836    0.21956  -0.311  0.75552    
post             0.02499    0.09677   0.258  0.79621    


Observed-attrition SA — Condition identification (unadjusted)— omnibus post × study_arm interaction (LRT, ML)
Data: long_condition
Models:
reduced_model: correct ~ post + study_arm + (1 | prolific_id) + (1 | vignette)
full_model: correct ~ post * study_arm + (1 | prolific_id) + (1 | vignette)
              npar   AIC   BIC  logLik -2*log(L) Chisq Df Pr(>Chisq)    
reduced_model    7 10817 10866 -5401.6     10803                        
full_model      10 10769 10839 -5374.3     10749 54.48  3  8.866e-12 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1


### Fit the unadjusted triage model

In [ ]:
triage_unadj <- glmer(
  correct ~ post * study_arm + (1 | prolific_id) + (1 | vignette),
  data = long_triage,
  family = binomial,
  control = glmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 2e5))
)

print_post_terms(
  triage_unadj,
  "Observed-attrition SA — Triage (unadjusted)"
)
print_post_study_arm_omnibus(
  triage_unadj,
  "Observed-attrition SA — Triage (unadjusted)"
)


Observed-attrition SA — Triage (unadjusted)
N participants: 861 ; N observations: 8431 
Generalized linear mixed model fit by maximum likelihood (Laplace
  Approximation) [glmerMod]
 Family: binomial  ( logit )
Formula: correct ~ post * study_arm + (1 | prolific_id) + (1 | vignette)
   Data: long_triage
Control: glmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 2e+05))

      AIC       BIC    logLik -2*log(L)  df.resid 
   8088.4    8158.8   -4034.2    8068.4      8421 

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-9.6326 -0.5127  0.1591  0.6158  3.3943 

Random effects:
 Groups      Name        Variance Std.Dev.
 prolific_id (Intercept) 0.5087   0.7132  
 vignette    (Intercept) 3.3679   1.8352  
Number of obs: 8431, groups:  prolific_id, 861; vignette, 5

Fixed effects:
                Estimate Std. Error z value Pr(>|z|)    
(Intercept)      1.00521    0.81716   1.230   0.2187    
post            -0.05765    0.11102  -0.519   0.6036    
study_arm1      -0.1


Observed-attrition SA — Triage (unadjusted)— omnibus post × study_arm interaction (LRT, ML)
Data: long_triage
Models:
reduced_model: correct ~ post + study_arm + (1 | prolific_id) + (1 | vignette)
full_model: correct ~ post * study_arm + (1 | prolific_id) + (1 | vignette)
              npar    AIC    BIC  logLik -2*log(L)  Chisq Df Pr(>Chisq)    
reduced_model    7 8101.6 8150.9 -4043.8    8087.6                         
full_model      10 8088.4 8158.8 -4034.2    8068.4 19.199  3  0.0002487 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1


## Tipping point for Perplexity non-completers

The models above already include every assigned participant with observed cells (805 completers + 56 non-completers). This section does not replace them.

Here the sample is the same 861 assigned participants as the models above, with every observed cell kept. Only missing post cells of Perplexity participants who did not finish are filled. Each of those cells is set to incorrect (0) for a share p; the remaining missing cells stay excluded. p is 0, 0.25, 0.50, 0.75, and 1. The number of cells set to 0 is `round(p × n missing)`. At p of 0 or 1 the fill is fixed, so the model is fit once. Otherwise 200 draws are taken (`TIP_REPS` overrides that), with seed `20260430 + draw` for condition identification and `20260530 + draw` for triage. At p = 0 the Perplexity × time term is the model fitted above.

The reported Perplexity × time term is `post:study_arm3` versus control. Log odds ratios are combined with Rubin's rules. The odds ratio and Wald interval use the pooled estimate and its total standard error. `median_p` is the median of the draw-level Wald p values.

In [ ]:
library(parallel)

N_REPS <- as.integer(Sys.getenv("TIP_REPS", unset = "200"))
if (is.na(N_REPS) || N_REPS < 1) stop("TIP_REPS must be a positive integer")
P_GRID <- c(0, 0.25, 0.50, 0.75, 1)
SEED_BY_OUTCOME <- c(condition = 20260430, triage = 20260530)
RNGkind("Mersenne-Twister", "Inversion", "Rejection")

tip_df <- df %>%
  filter(sample_group %in% c("analysed", "allocated_incomplete"))
cat(
  "Tipping-point sample:", nrow(tip_df),
  "(", sum(tip_df$sample_group == "analysed"), "completers +",
  sum(tip_df$sample_group == "allocated_incomplete"), "assigned non-completers)\n"
)

missing_post_rows <- function(observed, post_cols) {
  bind_rows(lapply(seq_along(VIGNETTES), function(i) {
    tip_df %>%
      filter(
        sample_group == "allocated_incomplete",
        as.character(study_arm) == "3",
        is.na(.data[[post_cols[i]]])
      ) %>%
      transmute(
        prolific_id = factor(as.character(prolific_id), levels = levels(observed$prolific_id)),
        vignette = factor(VIGNETTES[i], levels = levels(observed$vignette)),
        study_arm = factor(as.character(study_arm), levels = levels(observed$study_arm)),
        sample_group = "allocated_incomplete",
        country = as.character(.data[[COUNTRY_COL]]),
        Age = Age,
        Sex = factor(as.character(Sex), levels = levels(observed$Sex)),
        ai_frequency = as.numeric(ai_frequency),
        ai_literacy_sum = as.numeric(ai_literacy_sum),
        post = 1L,
        correct = 0L
      )
  }))
}

pool_rubin <- function(estimates, variances) {
  m <- length(estimates)
  qbar <- mean(estimates)
  ubar <- mean(variances)
  between <- if (m < 2) 0 else stats::var(estimates)
  total <- ubar + (1 + 1 / m) * between
  se <- sqrt(total)
  z <- qbar / se
  c(estimate = qbar, se = se, p = 2 * pnorm(-abs(z)))
}

fit_draw <- function(rep_id, observed, missing_rows, k, seed_base, ctrl, arm_ref) {
  RNGkind("Mersenne-Twister", "Inversion", "Rejection")
  set.seed(seed_base + rep_id)
  chosen <- if (k == 0) {
    missing_rows[0, , drop = FALSE]
  } else {
    missing_rows[sample.int(nrow(missing_rows), k), , drop = FALSE]
  }
  data <- dplyr::bind_rows(observed, chosen)
  data$study_arm <- stats::relevel(factor(data$study_arm), ref = arm_ref)
  model <- tryCatch(
    suppressWarnings(lme4::glmer(
      correct ~ post * study_arm + (1 | prolific_id) + (1 | vignette),
      data = data,
      family = stats::binomial,
      control = ctrl
    )),
    error = function(e) e
  )
  if (inherits(model, "error")) {
    message("Draw ", rep_id, " failed: ", conditionMessage(model))
    return(data.frame(
      rep = rep_id, failed = TRUE, message = conditionMessage(model),
      estimate = NA_real_, se = NA_real_, p = NA_real_,
      n_rows = NA_integer_, n_participants = NA_integer_, converged = FALSE,
      stringsAsFactors = FALSE
    ))
  }
  coef_mat <- stats::coef(summary(model))
  if (!"post:study_arm3" %in% rownames(coef_mat)) {
    stop("post:study_arm3 was not in the model")
  }
  row <- coef_mat["post:study_arm3", ]
  messages <- model@optinfo$conv$lme4$messages
  data.frame(
    rep = rep_id,
    failed = FALSE,
    message = NA_character_,
    estimate = unname(row[["Estimate"]]),
    se = unname(row[["Std. Error"]]),
    p = unname(row[["Pr(>|z|)"]]),
    n_rows = nrow(model@frame),
    n_participants = dplyr::n_distinct(model@frame$prolific_id),
    converged = length(messages) == 0,
    stringsAsFactors = FALSE
  )
}
environment(fit_draw) <- baseenv()

run_p <- function(observed, missing_rows, p_value, seed_base) {
  k <- as.integer(round(p_value * nrow(missing_rows)))
  n_reps <- if (k == 0 || k == nrow(missing_rows)) 1L else N_REPS
  cat(
    "  p =", p_value, ":", k, "of", nrow(missing_rows),
    "missing post cells set to 0;", n_reps, "fit(s)\n"
  )
  if (n_reps == 1L) {
    draws <- fit_draw(0L, observed, missing_rows, k, seed_base, GLMER_CTRL, ARM_REF)
  } else {
    n_cores <- max(1L, min(8L, parallel::detectCores(logical = TRUE) - 1L))
    cl <- parallel::makeCluster(n_cores)
    on.exit(parallel::stopCluster(cl), add = TRUE)
    parallel::clusterEvalQ(cl, {
      RNGkind("Mersenne-Twister", "Inversion", "Rejection")
      NULL
    })
    worker <- function(rep_id) {
      fit_draw(rep_id, observed, missing_rows, k, seed_base, ctrl, arm_ref)
    }
    worker_env <- new.env(parent = baseenv())
    worker_env$observed <- observed
    worker_env$missing_rows <- missing_rows
    worker_env$k <- k
    worker_env$seed_base <- seed_base
    worker_env$ctrl <- GLMER_CTRL
    worker_env$arm_ref <- ARM_REF
    worker_env$fit_draw <- fit_draw
    environment(worker) <- worker_env
    draws <- dplyr::bind_rows(parallel::parLapply(cl, seq_len(n_reps), worker))
  }
  ok <- draws[!draws$failed, , drop = FALSE]
  if (nrow(ok) == 0) stop("Every draw failed at p = ", p_value)
  pooled <- pool_rubin(ok$estimate, ok$se^2)
  data.frame(
    p_set_to_wrong = p_value,
    n_missing_post_cells = nrow(missing_rows),
    n_cells_set_to_zero = k,
    n_reps = nrow(ok),
    n_failed = sum(draws$failed),
    n_not_converged = sum(!ok$converged),
    n_rows = ok$n_rows[1],
    n_participants = ok$n_participants[1],
    log_or = unname(pooled[["estimate"]]),
    se_log_or = unname(pooled[["se"]]),
    or = exp(unname(pooled[["estimate"]])),
    ci_low = exp(unname(pooled[["estimate"]]) - 1.96 * unname(pooled[["se"]])),
    ci_high = exp(unname(pooled[["estimate"]]) + 1.96 * unname(pooled[["se"]])),
    p_rubin = unname(pooled[["p"]]),
    median_p = median(ok$p),
    stringsAsFactors = FALSE
  )
}
outcomes <- list(
  condition = list(pre = PRE_CONDITION_COLS, post = POST_CONDITION_COLS),
  triage = list(pre = PRE_NEXTSTEP_COLS, post = POST_NEXTSTEP_COLS)
)

EXPECTED_P0 <- c(condition = 0.41699, triage = 0.34102)
summary_rows <- list()
for (outcome_name in names(outcomes)) {
  cat("\n", outcome_name, "\n", sep = "")
  spec <- outcomes[[outcome_name]]
  observed <- make_long_outcome(tip_df, spec$pre, spec$post)
  missing_rows <- missing_post_rows(observed, spec$post)
  seed_base <- SEED_BY_OUTCOME[[outcome_name]]
  cat("  observed rows:", nrow(observed), "; missing post cells:", nrow(missing_rows), "\n")
  for (p_value in P_GRID) {
    piece <- dplyr::bind_cols(
      data.frame(outcome = outcome_name, stringsAsFactors = FALSE),
      run_p(observed, missing_rows, p_value, seed_base)
    )
    if (p_value == 0) {
      if (piece$n_missing_post_cells != 95 || piece$n_participants != 861) {
        stop(
          outcome_name, " p = 0 has n_missing = ", piece$n_missing_post_cells,
          " and n_participants = ", piece$n_participants
        )
      }
      if (abs(piece$log_or - EXPECTED_P0[[outcome_name]]) > 0.005) {
        stop(
          outcome_name, " p = 0 log-OR ", piece$log_or,
          " does not match the 861-participant model (",
          EXPECTED_P0[[outcome_name]], ")"
        )
      }
    }
    summary_rows[[paste(outcome_name, p_value)]] <- piece
  }
}

tipping_summary <- bind_rows(summary_rows)
cat("\nTipping point: Perplexity × time versus control\n")
print(as.data.frame(tipping_summary), row.names = FALSE, digits = 4)

for (outcome_name in c("condition", "triage")) {
  sub <- tipping_summary %>% filter(outcome == outcome_name)
  lost <- sub %>% filter(p_rubin >= 0.05)
  if (nrow(lost) == 0) {
    cat(outcome_name, ": still p < 0.05 when every missing post cell is set to incorrect\n")
  } else {
    cat(
      outcome_name, ": first grid point with Rubin p >= 0.05 is p =",
      lost$p_set_to_wrong[1], "\n"
    )
  }
}

SUMMARY_PATH <- file.path(ROOT, "output", "perplexity_tipping_point.csv")
write.csv(tipping_summary, SUMMARY_PATH, row.names = FALSE)
cat("\nSaved", SUMMARY_PATH, "\n")